In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from typing import Type
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from datasets import load_dataset

from latentqa.lqa_data import LQAData, load_lqa_data
from user_modeling.user_attribute import Gender, Country, Religion, Occupation
from latentqa.blogpost import (
    process_dataset,
    get_everything_needed_for_reading_plots,
    TEST_SYNTHSYS_ATTR_CLASSES,
    SELF_DESCRIBE_ATTR_CLASSES,
    PRISM_ATTR_CLASSES,
    plot_panel,
    plot_steering_for_attr_pairs,
)
from latentqa.eval.load_results import (
    load_steering_eval_results,
    load_circuit_based_steering_eval_results,
)

BASE_DIR = Path("") # <- Set this to where you downloaded the ...
DATA_DIR = Path(BASE_DIR / "llama31_8b_data")

# Reading

In [ ]:
EXPS_TO_LOAD: dict[str, list[str] | None] | None = {
    "llama31_8b_experiments": [
        "15_to_0",
        "all_to_all",
        "15_to_0_noheldout",
        "all_to_all_noheldout",
    ],
}

KWARGS = {
    "15_to_0": {
        "label": r"$15 \to 0$ (Excl. Held-out)",
        "color": "tab:blue",
        "linestyle": "-",
        "marker": "o",
        "markersize": 4.5,
    },
    "all_to_all": {
        "label": r"$\mathrm{all} \to \mathrm{all}$ (Excl. Held-out)",
        "color": "tab:blue",
        "linestyle": "--",
        "marker": "*",
        "markersize": 5,
    },
    "15_to_0_noheldout": {
        "label": r"$15 \to 0$ (Incl. Held-out)",
        "color": "tab:green",
        "linestyle": "-",
        "marker": "o",
        "markersize": 4.5,
    },
    "all_to_all_noheldout": {
        "label": r"$\mathrm{all} \to \mathrm{all}$ (Incl. Held-out)",
        "color": "tab:green",
        "linestyle": "--",
        "marker": "*",
        "markersize": 5,
    },
}

In [ ]:
# Get the model beliefs to use as labels for the gender split of PRISM.
hf_dataset = load_dataset("Transluce/PRISM-gender-Llama-3.1-8B-Instruct", split="train")
prism_gender_labels_by_tag = {}
for d in hf_dataset:
    attr = Gender(d["attr"])
    tag = d["tag"]
    prism_gender_labels_by_tag[tag] = attr

In [ ]:
# for dataset in ["synthsys", "selfdescribe", "prism"]:
#     process_dataset(dataset, DATA_DIR, overwrite=True)

In [ ]:
args_by_dataset_name = {}
for dataset_name in ["synthsys", "selfdescribe", "prism"]:
    results_df, tags_by_attr_class = get_everything_needed_for_reading_plots(
        data_dir=DATA_DIR, dataset_name=dataset_name, exps_to_load=EXPS_TO_LOAD
    )
    args_by_dataset_name[dataset_name] = (results_df, tags_by_attr_class)

In [ ]:
dataset_name_and_attr_classes = [
    ("synthsys", "SynthSys (Gender)", [Gender]),
    ("selfdescribe", "SelfDescribe (Gender)", [Gender]),
    ("prism", "PRISM (Gender)", [Gender]),
    (
        "synthsys",
        "SynthSys (Non-Gender)",
        [ac for ac in TEST_SYNTHSYS_ATTR_CLASSES if ac != Gender],
    ),
    (
        "selfdescribe",
        "SelfDescribe (Non-Gender)",
        [ac for ac in SELF_DESCRIBE_ATTR_CLASSES if ac != Gender],
    ),
    ("prism", "PRISM (Non-Gender)", [ac for ac in PRISM_ATTR_CLASSES if ac != Gender]),
]

exp_names = [
    "all_to_all_s32",
    "15_to_0",
    "all_to_all",
    "15_to_0_noheldout",
    "all_to_all_noheldout",
]

fig, axes = plt.subplots(2, 3, figsize=(4 * 3, 3 * 2))
for row in range(2):
    for col in range(3):
        dataset_name, title, attr_classes = dataset_name_and_attr_classes[row * 3 + col]
        results_df_, tags_by_attr_class_ = args_by_dataset_name[dataset_name]

        kwargs = KWARGS.copy()

        if title == "PRISM (Gender)":
            labels_by_tag = prism_gender_labels_by_tag
        else:
            labels_by_tag = None

        plot_panel(
            ax=axes[row, col],
            exp_names=exp_names,
            results_df=results_df_,
            tags_by_attr_class=tags_by_attr_class_,
            attr_classes=attr_classes,
            kwargs_by_exp_name=kwargs,
            labels_by_tag=labels_by_tag,
            ignore_label=True if (row == 1 and col == 1) else False,
        )
        axes[row, col].set_title(title)

legend_elements = []
for key, value in kwargs.items():
    legend_elements.append(Line2D([0], [0], **value))

# Create a single legend at the bottom center
fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    frameon=True,
    bbox_to_anchor=(0.5, -0.1),
)

fig.tight_layout()

# Steering

In [ ]:
all_results_df = load_steering_eval_results(BASE_DIR / "llama31_8b_steering_results")

In [ ]:
attr_pairs = []
attr_pairs_by_attr_class = defaultdict(list)
for (true, target, attr_class), _ in all_results_df.groupby(
    ["true_attr", "target_attr", "attr_class"], sort=False
):
    attr_pairs_by_attr_class[attr_class].append((attr_class(true), attr_class(target)))
    attr_pairs.append((attr_class(true), attr_class(target)))

In [ ]:
attr_class = Country  # [Country, Religion, Occupation]
control_prefix = "gen"
all_to_all = True

plot_steering_for_attr_pairs(
    all_results_df,
    steering_type="gradient",
    exp_name="all_to_all" if all_to_all else "15_to_0",
    seed=64,
    ckpt_name="step_4999",
    attr_pairs=attr_pairs_by_attr_class[attr_class],
    metric="prob",
    control_prefix=control_prefix,
    mean_kwargs={},
    sharey=True,
    title=(
        r"Steering using $\mathrm{all} \to \mathrm{all}$ Decoder" + f" ({attr_class.__name__})"
        if all_to_all
        else r"Steering using $15 \to 0$ Decoder" + f" ({attr_class.__name__})"
    ),
    title_fontsize=17,
    title_y=0.995,
)

# Circuit-Based Steering

In [ ]:
all_results_df = load_circuit_based_steering_eval_results(
    BASE_DIR / "llama31_8b_circuit_steering_results"
)

In [ ]:
target_attr = Country.INDIA
all_to_all = True
target_attr_str = target_attr.value_name.lower().split(" ")[-1]

if target_attr == Country.FRANCE:
    attr_pairs = [
        (Country.NEW_ZEALAND, Country.FRANCE),
        (Country.ESTONIA, Country.FRANCE),
        (Country.JORDAN, Country.FRANCE),
        (Country.GERMANY, Country.FRANCE),
        (Country.TAIWAN, Country.FRANCE),
        (Country.ITALY, Country.FRANCE),
        (Country.MEXICO, Country.FRANCE),
    ]
elif target_attr == Country.INDIA:
    attr_pairs = [
        (Country.INDONESIA, Country.INDIA),
        (Country.DENMARK, Country.INDIA),
        (Country.ICELAND, Country.INDIA),
        (Country.FRANCE, Country.INDIA),
        (Country.SOUTH_AFRICA, Country.INDIA),
        (Country.UNITED_ARAB_EMIRATES, Country.INDIA),
        (Country.NORWAY, Country.INDIA),
    ]
elif target_attr == Country.SOUTH_KOREA:
    attr_pairs = [
        (Country.PARAGUAY, Country.SOUTH_KOREA),
        (Country.JORDAN, Country.SOUTH_KOREA),
        (Country.NORWAY, Country.SOUTH_KOREA),
        (Country.SOUTH_AFRICA, Country.SOUTH_KOREA),
        (Country.ESTONIA, Country.SOUTH_KOREA),
        (Country.FRANCE, Country.SOUTH_KOREA),
        (Country.CHILE, Country.SOUTH_KOREA),
        (Country.TUNISIA, Country.SOUTH_KOREA),
        (Country.PARAGUAY, Country.SOUTH_KOREA),
    ]
else:
    raise ValueError(f"Target attribute {target_attr} not supported")

plot_steering_for_attr_pairs(
    all_results_df,
    steering_type="circuit",
    exp_name="all_to_all" if all_to_all else "15_to_0",
    ckpt_name="step_4999",
    seed=64,
    attr_pairs=attr_pairs,
    metric="prob",
    control_prefix="fixed",
    mean_kwargs={},
    title=r"Steering $a_\text{source} \to$ " + target_attr.value_name,
    title_fontsize=17,
    title_y=0.99,
)